# 08 Figures and tables

Renders the paper tables and figures from the frozen metric tables and the site-day table: the headline table, the Ausgrid presentation table, the method comparison, the per-station appendix, the review-burden figure, calibration, and 27 sample site-days per method.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** `07_metrics/*.csv`, `07_metrics/gate.json`, `05_m9/calibration_fits.csv`, `06_site_days/site_days.parquet`, the interval tables (for the sample panels) and the Beta dataset (recorded net load and solar of the sampled days).

**Outputs.** `outputs/01_final_evaluation/08_tables/*.csv|.md` and `08_figures/*.png`, `08_figures/samples_index.csv`; `manifests/08_report.json`.

**Approximate runtime.** About three minutes.

**Prerequisites.** Notebook 07.

**Main process.**

1. Write the tables as CSV and Markdown.
2. Draw the headline, per-station, review-burden, coverage and calibration figures.
3. Select and draw 27 Beta `sure` sample panels per method (TP 9, FN 6, FP 6, TN 6) by the seeded rule in `final_eval/figures.py`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Render

In [ ]:
paths = cli.stage_report(SETTINGS)
print(len(paths), "files written")

## 3. Tables

In [ ]:
for name in ["table01_headline", "table02_ausgrid", "table03_method_comparison", "table04_station_energy_iou"]:
    display(Markdown(f"### {name}"))
    display(Markdown((OUT / "08_tables" / f"{name}.md").read_text(encoding="utf-8")))

## 4. Figures

The review-burden figure reads the confidence-versus-coverage table: as c rises, fewer days are decided automatically, more go to review, and the automatic decisions carry fewer errors.

In [ ]:
for name in ["fig01_headline_metrics", "fig02_station_energy_iou", "fig03_station_energy_precision", "fig04_m9_review_burden_beta",
             "fig06_m9_coverage_scores_beta", "fig07_m9_calibration"]:
    display(Image(filename=OUT / "08_figures" / f"{name}.png"))

## 5. Sample site-days

Each panel shows the recorded net load, the solar estimate, the underlying load if the sign is kept (solar plus net load) and if corrected, the method's window and the reference window.

In [ ]:
for method in ["m9", "m8", "m7"]:
    for kind in ["TP", "FN", "FP", "TN"]:
        path = OUT / "08_figures" / f"samples_{method}_{kind}.png"
        if path.exists():
            display(Image(filename=path))

## Conclusion

Every table and figure in the paper traces to a file in `08_tables/` or `08_figures/` and, through the manifests, to the code, configuration and data that produced it.